In [68]:
# import os

# os.environ["REQUESTS_CA_BUNDLE"] = "./certs/pwc-bundle.pem"
# os.environ["SSL_CERT_FILE"] = "./certs/pwc-bundle.pem"

In [69]:
import pandas as pd

pd.set_option("display.max_colwidth", None)   # show full cell content
pd.set_option("display.max_rows", None)       # optional, show all rows

In [70]:
import pandas as pd
import numpy as np

ID_COL = "TalentLink ID"

df = pd.read_excel("../data/outputs/synthetic talent link data.xlsx", header=6)

# Clean blanks only (no row deletion except fully empty)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.dropna(how="all")

# Trim strings
obj_cols = df.select_dtypes(include=["object", "string"]).columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.astype("string").str.strip())

# ID as numeric (important for grouping)
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

# Parse job dates safely
for c in ["Job History Start Date", "Job History End Date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df.head(2)

,Local employee ID,TalentLink ID,Resource Name,Resource Status,Resource Type,Management level,Job level,Grade Code,Global LoS,Global Network,...,Area of Study,Academic Institution,In Process,Project Type Interest,Travel Interest,Additional Information,Profile Last Edited (YYYY/MM/DD),Unnamed_BC,Unnamed_BD,Unnamed_BE
0,104332181,14265799,Allison Hill,Enabled,Associate,Director,Senior Manager,A2,Advisory,Technology,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2025/03/07,r104332181,45712,Active TalentLink filter
1,104332181,14265799,Allison Hill,Enabled,Associate,Director,Senior Manager,A2,Advisory,Technology,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2025/03/07,r104332181,45712,Active TalentLink filter


In [71]:
skills_per_id = df[df["Skill Level 1"].isin(["Skills"])]

skills_per_id_grouped = (
    skills_per_id.groupby("TalentLink ID", sort=False)["Capability Name"]
        .agg(list)
        .reset_index(name="skills")
)

skills_per_id_grouped.head()

,TalentLink ID,skills
0,14265799,"[Data Analysis, Risk Controls, Python, Risk Controls, Regulatory Reporting, Programme Delivery, Regulatory Reporting]"
1,70217,"[ETL Design, ETL Design, Python, SQL, Data Governance, Data Governance]"
2,271339,"[Python, Python, Data Governance, Data Governance, Python, Regulatory Reporting, Regulatory Reporting, Data Analysis, Power BI, Data Governance, Risk Controls, Regulatory Reporting, ETL Design, Python, Data Governance, Regulatory Reporting, Data Analysis, Data Governance, Data Analysis, Programme Delivery, Python, Programme Delivery, Risk Controls, Python, Regulatory Reporting, Stakeholder Management, Programme Delivery, Python, Stakeholder Management, Regulatory Reporting, Risk Controls, Risk Controls, Data Governance, ETL Design]"
3,88320463,"[Regulatory Reporting, ETL Design, Stakeholder Management, Regulatory Reporting, Programme Delivery, ETL Design, Data Governance, Data Analysis, Power BI, SQL, Regulatory Reporting, Power BI, Python]"
4,10435578,"[Power BI, ETL Design, Risk Controls]"


In [72]:
#This works
language = df[df["Skill Level 1"].isin(["Language"])]

langauage_grouped = (
    language.groupby("TalentLink ID", sort=False)["Capability Name"]
    .agg(list)
    .reset_index(name="languages")
)

langauage_grouped.head()

,TalentLink ID,languages
0,14265799,"[English, Italian]"
1,70217,"[English, Spanish]"
2,271339,[English]
3,88320463,[English]
4,10435578,"[English, German]"


In [73]:
#This works
summary = df.drop_duplicates(subset=['TalentLink ID', 'Summary/bio'])

biography = (
    summary
        .groupby("TalentLink ID", sort=False)["Summary/bio"]
        .agg(list)
        .reset_index(name="Biography")
)

biography[biography["TalentLink ID"] == 97817]

,TalentLink ID,Biography


In [74]:
import pandas as pd
from itertools import zip_longest

# --- 0. columns we care about
cols = [
    "Job History Start Date",
    "Job History End Date",
    "Position",
    "Company",
    "Description"
]

# --- 1. select, dedupe, and copy
jobs_by_id = (
    df[[
        "TalentLink ID",
        *cols
    ]]
    .dropna(subset=["Description"])
    .drop_duplicates()
    .copy()
)

# --- 2. clean text columns safely
text_cols = ["Position", "Company", "Description"]
for c in text_cols:
    jobs_by_id.loc[:, c] = (
        jobs_by_id[c].astype(str)
        .str.replace(r"\n+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": None})
    )

# --- 3. parse & format dates keeping gaps as None
for c in ["Job History Start Date", "Job History End Date"]:
    jobs_by_id.loc[:, c] = pd.to_datetime(jobs_by_id[c], errors="coerce")
    formatted = jobs_by_id[c].dt.strftime("%Y-%m-%d")
    jobs_by_id.loc[:, c] = formatted.where(jobs_by_id[c].notna(), None)

# --- 4. optional sort so arrays are chronological
jobs_by_id = jobs_by_id.sort_values(
    ["TalentLink ID", "Job History Start Date", "Job History End Date"],
    na_position="last"
)

# --- 5. make parallel lists per ID
grouped = (
    jobs_by_id
      .groupby("TalentLink ID", sort=False)[cols]
      .agg(list)
      .reset_index()
)

# --- 6. convert parallel lists -> list-of-dicts (aligned; zip_longest protects against mismatched lengths)
def lists_to_jobs(row):
    lists = [row[c] for c in cols]
    jobs = []
    for start, end, pos, comp, desc in zip_longest(*lists, fillvalue=None):
        jobs.append({
            "Job History Start Date": start,
            "Job History End Date": end,
            "Position": pos,
            "Company": comp,
            "Description": desc
        })
    return jobs

grouped["Jobs"] = grouped.apply(lists_to_jobs, axis=1)

# --- 7. final frame: one row per ID with Jobs = list of aligned job dicts
projects = grouped[["TalentLink ID", "Jobs"]]

# explode for everyone while keeping TalentLink ID
person_jobs = projects.explode("Jobs").reset_index(drop=True)

# normalize the dict column (skip rows where Jobs is missing)
jobs_df = pd.concat(
    [
        person_jobs[["TalentLink ID"]],
        pd.json_normalize(person_jobs["Jobs"])
    ],
    axis=1
)


I think i should join up all the datasets now to a new big dataset.

In [75]:
master_dataset = (
    skills_per_id_grouped.set_index("TalentLink ID")
    .join(biography.set_index("TalentLink ID"))
    .join(jobs_df.set_index("TalentLink ID"))
    .dropna(subset=["skills", "Description"])
)

In [76]:
master_dataset.to_csv("../data/outputs/master_dataset.csv", index=True)

In [77]:
#This works
riskTeams = df[['TalentLink ID', 'Strategic Region']]
team_groups = riskTeams.drop_duplicates(subset=['TalentLink ID', 'Strategic Region'])


team = (
    team_groups
        .groupby("TalentLink ID", sort=False)["Strategic Region"]
        .agg(list)
        .reset_index(name="Team")
)

team.head(3)


,TalentLink ID,Team
0,14265799,[Risk South]
1,70217,[Risk North]
2,271339,[Risk Central]


Now i think i have all the data i really need to build the machine learning model. Its time to start small. I need to find a way to transform text into numbers. This means i need to look at the projects and biography vairables and see if i can extract more info

I think i will use the 4 steps i found in my preliminary research

In [78]:
master_dataframe = (
    master_dataset
    .groupby("TalentLink ID")
    .agg({
        "skills": "first",
        "Biography": "first",
        "Job History Start Date": list,
        "Job History End Date": list,
        "Position": list,
        "Company": list,
        "Description": list
    })
    .reset_index()
)

In [79]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

nlp = spacy.load("en_core_web_sm")


Stop word reomval

In [80]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# load spaCy model
nlp = spacy.load("en_core_web_sm")

# =========================================================
# STEP 1: spaCy stop word removal
# =========================================================
def remove_stopwords(text):
    text = str(text)
    doc = nlp(text)
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

master_dataframe["step1_stopwords_removed"] = master_dataframe["Description"].apply(remove_stopwords)

# =========================================================
# STEP 2: spaCy POS tagging + lemmatization
# =========================================================
def tag_and_lemmatize(text):
    text = str(text)
    doc = nlp(text)

    lemmas = []
    pos_tags = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            lemmas.append(token.lemma_)
            pos_tags.append((token.text, token.pos_))

    return pd.Series([
        " ".join(lemmas),   # cleaned lemmatized text
        pos_tags            # token + POS tags
    ])

master_dataframe[["step2_lemmatized", "pos_tags"]] = master_dataframe["step1_stopwords_removed"].apply(tag_and_lemmatize)

# =========================================================
# STEP 3: TF-IDF vector representation
# =========================================================
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(master_dataframe["step2_lemmatized"])

# Optional: group by TalentLink ID
grouped_df = (
    master_dataframe
    .groupby("TalentLink ID")["step2_lemmatized"]
    .apply(" ".join)
    .reset_index()
)

X_grouped_tfidf = tfidf_vectorizer.fit_transform(grouped_df["step2_lemmatized"])

In [81]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# group employee text
employee_df = (
    master_dataframe
    .groupby("TalentLink ID", as_index=False)
    .agg({
        "step2_lemmatized": " ".join,
        "skills": "first"
    })
)

# create TF-IDF matrix
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform(employee_df["step2_lemmatized"])
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
tfidf_df["TalentLink ID"] = employee_df["TalentLink ID"]

# score only the employee's listed skills
skill_scores = []

for i, row in employee_df.iterrows():
    employee_id = row["TalentLink ID"]
    employee_skills = row["skills"]

    if not isinstance(employee_skills, list):
        employee_skills = []

    for skill in employee_skills:
        skill_lower = str(skill).lower()
        score = tfidf_df.loc[i, skill_lower] if skill_lower in tfidf_df.columns else 0

        skill_scores.append({
            "TalentLink ID": employee_id,
            "skill": skill,
            "score": score
        })

skill_scores_df = pd.DataFrame(skill_scores)

In [82]:
# keep only skills that appear in descriptions
filtered = skill_scores_df[skill_scores_df["score"] > 0]

skill_analysis_table = (
    filtered
    .groupby("skill")
    .agg(
        employees_demonstrating=("TalentLink ID", "nunique"),
        average_score=("score", "mean"),
        max_score=("score", "max")
    )
    .sort_values("employees_demonstrating", ascending=False)
    .reset_index()
)

skill_analysis_table.head()

,skill,employees_demonstrating,average_score,max_score


What i need to do here is add biography/summary as part of the calculation 

In [83]:
filtered.head()

,TalentLink ID,skill,score
